### Dependencies

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")

### Configuration

In [2]:
DATA_PATH = "merged.csv"   
OUT_DIR   = "outputs"

# Features to extract from the CSV
FEATURES = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Fwd Packets/s",
    "Bwd Packets/s",
    "Min Packet Length",
    "Max Packet Length",
    "Packet Length Mean",
    "Packet Length Std",
    "Destination Port",
]

print("Config loaded. OUT_DIR:", OUT_DIR)

Config loaded. OUT_DIR: outputs


## Data Loading

In [3]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df_raw):,} rows from {DATA_PATH}")

df_raw.columns = df_raw.columns.str.strip()
for col in df_raw.select_dtypes("object").columns:
    df_raw[col] = df_raw[col].str.strip()

if "Label" in df_raw.columns:
        df_raw["label"] = (df_raw["Label"] != "BENIGN").astype(int)

print(f"Attack rate : {df_raw['label'].mean()*100:.1f}%")
df_raw.head(3)

Loaded 2,830,743 rows from merged.csv
Attack rate : 19.7%


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,label
0,80,38308,1,1,6,6,6,6,6.000000,0.000000,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
1,389,479,11,5,172,326,79,0,15.636364,31.449238,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
2,88,1095,10,6,3150,3150,1575,0,315.000000,632.561635,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0


## Feature Engineering 

Steps:
1. Select a focused feature set (flows, bytes, packets, ports)  
2. Replace `Inf`/`NaN` with KNN  
3. Min-Max scale to **[0, 1]**  
4. Add **rolling mean & std** (window = 10 rows) to capture temporal bursts  
5. **Sliding-window segmentation** → shape `(N, seq_len, n_features)` for the LSTM

In [4]:
def engineer_features(df, feature_cols):
    available = [c for c in feature_cols if c in df.columns]
    if not available:
        available = [c for c in df.columns if c.startswith("feat_")][:len(feature_cols)]
    print(f"Using {len(available)} features")

    X = df[available].copy()
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    X.fillna(X.median(numeric_only=True), inplace=True)

    scaler  = MinMaxScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=available, index=df.index)

    # Rolling temporal features (window = 10 steps)
    for col in available[:4]:
        X_scaled[f"{col}_rmean"] = X_scaled[col].rolling(10, min_periods=1).mean()
        X_scaled[f"{col}_rstd"]  = X_scaled[col].rolling(10, min_periods=1).std().fillna(0)

    df_out = X_scaled.copy()
    df_out["label"]     = df["label"].values
    df_out["Label"]     = df["Label"].values
    if "Timestamp" in df.columns:
        df_out["Timestamp"] = df["Timestamp"].values
    return df_out, scaler


def make_sequences(data, seq_len):
    """Sliding window → (N, seq_len, n_features)."""
    return np.array([data[i:i+seq_len] for i in range(len(data)-seq_len+1)],
                    dtype=np.float32)


df_feat, scaler = engineer_features(df_raw, FEATURES)
FEAT_COLS = [c for c in df_feat.columns if c not in ("label","Label","Timestamp")]

print(f"Feature matrix : {len(df_feat):,} rows × {len(FEAT_COLS)} columns")
df_feat[FEAT_COLS].describe().round(3)


Using 14 features
Feature matrix : 2,830,743 rows × 22 columns


,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Flow Bytes/s,Flow Packets/s,Fwd Packets/s,Bwd Packets/s,Min Packet Length,...,Packet Length Std,Destination Port,Flow Duration_rmean,Flow Duration_rstd,Total Fwd Packets_rmean,Total Fwd Packets_rstd,Total Backward Packets_rmean,Total Backward Packets_rstd,Total Length of Fwd Packets_rmean,Total Length of Fwd Packets_rstd
count,2830743.000,2830743.000,2830743.000,2830743.000,2830743.000,2830743.000,2830743.000,2830743.000,2830743.000,2830743.000,...,2830743.000,2830743.000,2830743.000,2830743.000,2830743.000,2830743.000,2830743.000,2830743.000,2830743.0,2830743.000
mean,0.123,0.000,0.000,0.000,0.000,0.113,0.345,0.021,0.003,0.011,...,0.062,0.123,0.123,0.106,0.000,0.000,0.000,0.000,0.0,0.000
std,0.280,0.003,0.003,0.001,0.003,0.011,0.042,0.083,0.019,0.017,...,0.134,0.279,0.220,0.149,0.001,0.003,0.001,0.003,0.0,0.001
min,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.000
25%,0.000,0.000,0.000,0.000,0.000,0.112,0.333,0.000,0.000,0.000,...,0.000,0.001,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.000
50%,0.000,0.000,0.000,0.000,0.000,0.112,0.333,0.000,0.000,0.001,...,0.005,0.001,0.015,0.015,0.000,0.000,0.000,0.000,0.0,0.000
75%,0.027,0.000,0.000,0.000,0.000,0.112,0.337,0.004,0.004,0.025,...,0.037,0.007,0.116,0.210,0.000,0.000,0.000,0.000,0.0,0.000
max,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,...,1.000,1.000,0.995,0.527,0.101,0.316,0.101,0.316,0.1,0.316


In [5]:
def engineer_features(df, feature_cols):
    # 1. Identify available features
    available = [c for c in feature_cols if c in df.columns]
    if not available:
        available = [c for c in df.columns if c.startswith("feat_")][:len(feature_cols)]
    print(f"Using {len(available)} features for KNN Imputation")

    # 2. Prepare numerical data for imputation
    X = df[available].copy()
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # 3. KNN Imputation
    # n_neighbors=5 is standard; note that KNN is computationally expensive on very large datasets
    imputer = KNNImputer(n_neighbors=5)
    X_imputed = imputer.fit_transform(X)
    X = pd.DataFrame(X_imputed, columns=available, index=df.index)
    
    print(f"Missing values after KNN imputation: {X.isnull().sum().sum()}")

    # 4. Scaling
    scaler = MinMaxScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=available, index=df.index)

    # 5. Rolling temporal features (window = 10 steps)
    for col in available[:4]:
        X_scaled[f"{col}_rmean"] = X_scaled[col].rolling(10, min_periods=1).mean()
        X_scaled[f"{col}_rstd"]  = X_scaled[col].rolling(10, min_periods=1).std().fillna(0)

    # 6. Recombine with labels and metadata
    df_out = X_scaled.copy()
    
    # Safely add categorical/target columns if they exist
    for target in ["label", "Label", "Timestamp"]:
        if target in df.columns:
            df_out[target] = df[target].values
            
    return df_out, scaler

# Execute the updated pipeline
df_feat, scaler = engineer_features(df_raw, FEATURES)
FEAT_COLS = [c for c in df_feat.columns if c not in ("label", "Label", "Timestamp")]

print(f"Feature matrix: {len(df_feat):,} rows × {len(FEAT_COLS)} columns")
print(df_feat[FEAT_COLS].describe().round(3))

Using 14 features for KNN Imputation
Missing values after KNN imputation: 0
Feature matrix: 2,830,743 rows × 22 columns
       Flow Duration  Total Fwd Packets  Total Backward Packets  \
count    2830743.000        2830743.000             2830743.000   
mean           0.123              0.000                   0.000   
std            0.280              0.003                   0.003   
min            0.000              0.000                   0.000   
25%            0.000              0.000                   0.000   
50%            0.000              0.000                   0.000   
75%            0.027              0.000                   0.000   
max            1.000              1.000                   1.000   

       Total Length of Fwd Packets  Total Length of Bwd Packets  Flow Bytes/s  \
count                  2830743.000                  2830743.000   2830743.000   
mean                         0.000                        0.000         0.113   
std                          0.00

In [7]:
# Download new Data
file_name = 'engineered_network_traffic.csv.gz'
df_feat.to_csv(file_name, index=False, compression='gzip')

print(f"Saved: {file_name}")


Saved: engineered_network_traffic.csv.gz
